# Customer Churn Analysis

Explore churn patterns by contract type and plan type, review cancellation reasons, and summarize findings.

**Data:** provide an approved, anonymized file at `data/cleaned_churn.csv` before running the notebook. The source data is intentionally not included in this public portfolio copy.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("data/cleaned_churn.csv")

# The dataset spells the churn column "chrun_flag"
df["chrun_flag"].value_counts(normalize=True).mul(100).round(1)

In [ ]:
#comparing the churn rate according to the contract type 
churn_by_contract = (
    df.groupby("contract_type")["chrun_flag"]
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

ax = churn_by_contract.plot(kind="bar", color="teal", rot=0)
ax.set_title("Churn rate by contract type")
ax.set_ylabel("Churn rate (%)")
ax.set_xlabel("Contract type")
plt.tight_layout()
plt.show()

In [ ]:
#let’s compare churn rates across plan types
churn_by_plan = (
    df.groupby("plan_type")["chrun_flag"]
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

ax = churn_by_plan.plot(kind="bar", color="slateblue", rot=0)
ax.set_title("Churn rate by plan type")
ax.set_ylabel("Churn rate (%)")
ax.set_xlabel("Plan type")
plt.tight_layout()
plt.show()

In [ ]:

heatmap_data = df.pivot_table(
    index="contract_type",
    columns="plan_type",
    values="chrun_flag",
    aggfunc="mean"
) * 100

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    vmin=0,
    vmax=100,
    cbar_kws={"label": "Churn rate (%)"}
)

plt.title("Churn rate by contract and plan type")
plt.xlabel("Plan type")
plt.ylabel("Contract type")
plt.tight_layout()
plt.show()

In [ ]:
# Overall customer metrics
total = len(df)
churned = df["chrun_flag"].sum()
churn_rate = df["chrun_flag"].mean() * 100

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Customer Churn Overview", fontsize=18, fontweight="bold")

# KPI summary
fig.text(
    0.5, 0.92,
    f"Customers: {total}     |     Churned: {churned}     |     Churn rate: {churn_rate:.1f}%",
    ha="center", fontsize=12
)

# Churn rate by contract type
contract = (
    df.groupby("contract_type")["chrun_flag"]
      .agg(["mean", "count", "sum"])
      .reset_index()
)
contract["churn_pct"] = contract["mean"] * 100

sns.barplot(data=contract, x="contract_type", y="churn_pct",
            color="teal", ax=axes[0, 0])
axes[0, 0].set(title="Churn by Contract Type",
               xlabel="Contract type", ylabel="Churn rate (%)")
axes[0, 0].set_ylim(0, 100)

for i, row in contract.iterrows():
    axes[0, 0].text(i, row["churn_pct"] + 2,
                    f'{row["churn_pct"]:.1f}% ({int(row["sum"])}/{int(row["count"])})',
                    ha="center")


# Churn rate by plan type
plan = (
    df.groupby("plan_type")["chrun_flag"]
      .agg(["mean", "count", "sum"])
      .reset_index()
)
plan["churn_pct"] = plan["mean"] * 100

sns.barplot(data=plan, x="plan_type", y="churn_pct",
            color="slateblue", ax=axes[0, 1])
axes[0, 1].set(title="Churn by Plan Type",
               xlabel="Plan type", ylabel="Churn rate (%)")
axes[0, 1].set_ylim(0, 100)

for i, row in plan.iterrows():
    axes[0, 1].text(i, row["churn_pct"] + 2,
                    f'{row["churn_pct"]:.1f}% ({int(row["sum"])}/{int(row["count"])})',
                    ha="center")

# Reasons given by churned customers
reasons = (
    df.loc[df["chrun_flag"] == 1, "cancellation_reason"]
      .dropna()
      .value_counts()
      .rename_axis("reason")
      .reset_index(name="customers")
)

sns.barplot(data=reasons, x="customers", y="reason",
            color="coral", ax=axes[1, 0])
axes[1, 0].set(title="Reported Cancellation Reasons",
               xlabel="Number of customers", ylabel="Reason")

# Monthly charges for retained and churned customers
sns.boxplot(data=df, x="chrun_flag", y="monthly_charges",
            color="lightblue", ax=axes[1, 1])
sns.stripplot(data=df, x="chrun_flag", y="monthly_charges",
              color="black", alpha=0.6, ax=axes[1, 1])
axes[1, 1].set(title="Monthly Charges by Churn Status",
               xlabel="Customer status", ylabel="Monthly charges")
axes[1, 1].set_xticks([0, 1], ["Stayed", "Churned"])

plt.tight_layout(rect=[0, 0, 1, 0.89])
plt.show()